In [1]:
# Imports and paths
import os
import pandas as pd
from Bio import SeqIO

deeptmhmm_dir = "../results/deeptmhmm/full_run"

input_fasta = (
    "../results/signalp_fast_slow/"
    "all_signalp_predictions.fasta"
)

output_fasta = (
    "../results/deeptmhmm/"
    "validated_secretome.fasta"
)

classification_csv = (
    "../results/deeptmhmm/"
    "deeptmhmm_classification.csv"
)

summary_csv = (
    "../results/deeptmhmm/"
    "deeptmhmm_summary.csv"
)

topology_file = os.path.join(
    deeptmhmm_dir,
    "predicted_topologies.3line"
)

In [2]:
# Parse DeepTMHMM classes
records = []

with open(topology_file) as f:

    while True:

        header = f.readline().strip()

        if not header:
            break

        sequence = f.readline().strip()
        topology = f.readline().strip()

        full_id = (
            header.split("| ")[0]
            .replace(">", "")
        )
        
        protein = full_id.split("|")[1]

        protein_class = (
            header.split("| ")[1]
            .strip()
        )

        records.append({
            "protein": protein,
            "class": protein_class
        })

classification = pd.DataFrame(records)

classification.head()

,protein,class
0,A0A1L9U2J4,SP
1,A0A1L9U2T6,SP
2,A0A1L9U5R7,SP
3,A0A1L9U6P4,SP
4,A0A1L9U6P8,SP


In [3]:
# Summary table
summary = (
    classification["class"]
    .value_counts()
    .rename_axis("class")
    .reset_index(name="proteins")
)

summary["percent"] = (
    summary["proteins"]
    / summary["proteins"].sum()
    * 100
)

summary

,class,proteins,percent
0,SP,19631,83.832259
1,SP+TM,3192,13.631123
2,GLOB,391,1.669727
3,TM,203,0.866892


In [4]:
# Save classification tables
classification.to_csv(
    classification_csv,
    index=False
)

summary.to_csv(
    summary_csv,
    index=False
)

print("Saved classification tables")

Saved classification tables


In [5]:
# Filter SP proteins only
sp_ids = set(
    classification.loc[
        classification["class"] == "SP",
        "protein"
    ]
)

print(
    f"SP proteins retained: "
    f"{len(sp_ids):,}"
)


SP proteins retained: 19,631


In [6]:
#Build validated secretome FASTA
secretome_records = []

for record in SeqIO.parse(
    input_fasta,
    "fasta"
):
    protein = record.id.split("|")[1]
    if protein in sp_ids:
        secretome_records.append(record)

print(
    f"Recovered sequences: "
    f"{len(secretome_records):,}"
)

SeqIO.write(
    secretome_records,
    output_fasta,
    "fasta"
)

print(
    f"Saved:\n{output_fasta}"
)

Recovered sequences: 19,631
Saved:
../results/deeptmhmm/validated_secretome.fasta


In [7]:
print(records[0])

{'protein': 'A0A1L9U2J4', 'class': 'SP'}


In [8]:
# Report
total = len(classification)

retained = (
    classification["class"] == "SP"
).sum()

removed = total - retained

print(f"Input proteins:   {total:,}")
print(f"Retained (SP):    {retained:,}")
print(f"Removed:          {removed:,}")
print(
    f"Retention: "
    f"{retained/total*100:.1f}%"
)

Input proteins:   23,417
Retained (SP):    19,631
Removed:          3,786
Retention: 83.8%
